# Lab 2: Train, compare, register, and use a milestone-delay model

Train six classifiers in two rounds in one AIDP experiment, compare their validation results in **Experiments** or in the notebook, register the selected model in the Master Catalog, then reload that exact catalog version to score Gold milestones.

Candidates are ranked by validation F1. When F1 is tied, the lower validation Brier score wins.

Allow 60–90 minutes, including compute startup. Experiments and Models are AIDP Preview features.

### What you will learn

Create an experiment; log parameters, metrics, and model artifacts; compare two model families; select a winner using validation F1 and Brier score; register it in the catalog; and use that registered version for inference.

| Stage | AIDP object or output |
| --- | --- |
| Both training rounds | Experiment `seer_milestone_delay` |
| Round one | Three Decision Tree runs: `dt_r1_1` to `dt_r1_3` |
| Round two | Three Gradient Boosting runs: `gb_r2_1` to `gb_r2_3` |
| Selected model | `seer_livelabs.seer_gold.milestone_delay_classifier` |
| Inference | Gold milestone scores from the selected catalog version |

### How to run this lab

Use workspace **seer_livelabs**. Run one Python cell at a time, from top to bottom, and wait for it to finish. Task numbers are Markdown headings, not Python execution counts. This version has **10 Python tasks** and ends after Task 10.

For the first walkthrough, stop after Task 6 to inspect the first model family before running Task 7.

## Prerequisites

1. Complete Lab 1A, Lab 1B, and Lab 1C in order.
2. Ensure Gold contains `milestone_training` and `milestone_features`.
3. Open **Shared → seer-aidp-livelabs → notebooks → Lab2_Predict_Investigate_and_Operationalize.ipynb**.
4. Attach **seer_livelabs_spark**. Use AMD/Intel compute, not ARM.
5. Use the platform-provided MLflow integration. Gradient Boosting is supplied by scikit-learn; no XGBoost installation is required.
6. Your identity needs access to workspace experiments and permission to register model versions in `seer_livelabs.seer_gold`.

`milestone_training` supplies the labeled history for model training. `milestone_features` supplies the current Gold rows for inference.

## Rerunning Lab 2

Task 1 creates a new `SESSION_ID`, so earlier runs remain separate from this walkthrough. Run Tasks 5 and 7 once per session. At Task 9, choose either the notebook cell or the Experiments UI to register the winner. Task 10 can be rerun to score the selected catalog version again.


## Task 1: Create the experiment

Run the cell to create or reuse `seer_milestone_delay`. It also defines the fully qualified model name and creates a session ID for this walkthrough.

**Checkpoint:** Open **Experiments → seer_milestone_delay**. It may be empty until Task 5.


In [ ]:
import json, os, uuid
from pathlib import Path

import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

LAB_HOME = Path(os.environ.get("SEER_LAB_HOME", "/Workspace/Shared/seer-aidp-livelabs"))
utilities = globals().get("oidlUtils")
config_path = str(LAB_HOME / "config/workshop.json")
if utilities is not None:
    config_path = utilities.parameters.getParameter("config_path", config_path)
config = json.loads(Path(config_path).read_text())

GOLD = f"{config['catalog']}.{config['schemas']['gold']}"
TRAINING_TABLE = f"{GOLD}.milestone_training"
SCORING_TABLE = f"{GOLD}.milestone_features"
MODEL_NAME = f"{GOLD}.milestone_delay_classifier"
FEATURES = ["days_to_planned_date", "open_po_fraction", "inspection_fail_flag",
            "inspection_info_flag", "supplier_review_flag", "unresolved_supplier_flag",
            "milestone_not_started_flag", "milestone_at_risk_flag", "log_committed_cost"]

experiment = mlflow.set_experiment("seer_milestone_delay")
mlflow.autolog(disable=True)
SESSION_ID = uuid.uuid4().hex
threshold = 0.5

print("Experiment:", experiment.name)
print("Session:", SESSION_ID)


## Task 2: Load Gold training history

Run the cell to load the labeled Gold training data and the nine model inputs.


In [ ]:
history = spark.table(TRAINING_TABLE).select(
    "milestone_id", "prediction_as_of", "late_flag", *FEATURES).toPandas()
history["prediction_as_of"] = pd.to_datetime(history["prediction_as_of"], utc=True)
history["late_flag"] = history["late_flag"].astype(int)
history[FEATURES] = history[FEATURES].astype(float)

print({"training_rows": len(history), "late_flag_counts": history.late_flag.value_counts().to_dict()})


## Task 3: Create the train/validation split

Run the cell to create a simple balanced split for the model comparison. Every candidate uses the same split.


In [ ]:
train, validation = train_test_split(
    history, test_size=0.5, random_state=42, stratify=history["late_flag"])

X_train, y_train = train[FEATURES], train.late_flag
X_validation, y_validation = validation[FEATURES], validation.late_flag
print({"training_rows": len(train), "validation_rows": len(validation)})


## Task 4: Define the experiment helper

Run the cell once. `train_and_log` creates one MLflow run and logs its validation metrics. `compare_runs` ranks this walkthrough's runs by validation F1, then validation Brier score.


In [ ]:
def validation_metrics(labels, probabilities):
    predicted = probabilities >= threshold
    return {
        "average_precision": average_precision_score(labels, probabilities),
        "roc_auc": roc_auc_score(labels, probabilities),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
        "brier_score": brier_score_loss(labels, probabilities),
    }

def train_and_log(run_name, algorithm, round_number, parameters):
    estimator_class = GradientBoostingClassifier if algorithm == "Gradient Boosting" else DecisionTreeClassifier
    estimator = estimator_class(random_state=42, **parameters)

    with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name) as run:
        mlflow.set_tags({"session_id": SESSION_ID, "round": str(round_number), "algorithm": algorithm})
        mlflow.log_params(parameters)
        estimator.fit(X_train, y_train)
        probabilities = estimator.predict_proba(X_validation)[:, list(estimator.classes_).index(1)]
        metrics = validation_metrics(y_validation, probabilities)
        mlflow.log_metrics({f"validation_{name}": value for name, value in metrics.items()})
        model_info = mlflow.sklearn.log_model(
            estimator, name="model", input_example=X_train.head(3),
            signature=infer_signature(X_train, estimator.predict_proba(X_train)),
            pyfunc_predict_fn="predict_proba")
        mlflow.set_tag("model_uri", model_info.model_uri)
        formatted_metrics = ", ".join(
            f"validation_{name} = {value:.4f}" for name, value in metrics.items()
        )
        print(f"{run_name}: {formatted_metrics}")
    return run.info.run_id

def compare_runs():
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.session_id = '{SESSION_ID}'")
    return runs.loc[runs.status == "FINISHED"].sort_values(
        ["metrics.validation_f1", "metrics.validation_brier_score", "tags.mlflow.runName"],
        ascending=[False, True, True]).reset_index(drop=True)

comparison_columns = [
    "run_id", "tags.mlflow.runName", "tags.algorithm",
    "metrics.validation_f1", "metrics.validation_brier_score",
    "metrics.validation_average_precision", "metrics.validation_roc_auc",
    "metrics.validation_precision", "metrics.validation_recall",
]

comparison_labels = {
    "run_id": "Run ID",
    "tags.mlflow.runName": "Run name",
    "tags.algorithm": "Algorithm",
    "metrics.validation_average_precision": "Validation average precision",
    "metrics.validation_roc_auc": "Validation ROC AUC",
    "metrics.validation_precision": "Validation precision",
    "metrics.validation_recall": "Validation recall",
    "metrics.validation_f1": "Validation F1",
    "metrics.validation_brier_score": "Validation Brier score (lower is better)",
}

def format_comparison(runs):
    return runs[comparison_columns].rename(columns=comparison_labels).round(4)


## Task 5: Round one — Decision Tree

Run the three Decision Tree candidates.

| Candidate | max_depth | min_samples_leaf |
| --- | --- | --- |
| dt_r1_1 | 3 | 10 |
| dt_r1_2 | 5 | 20 |
| dt_r1_3 | 8 | 30 |

Each configuration creates a separate run in the same experiment. Run this cell once.


In [ ]:
tree_candidates = [
    {"max_depth": 3, "min_samples_leaf": 10},
    {"max_depth": 5, "min_samples_leaf": 20},
    {"max_depth": 8, "min_samples_leaf": 30},
]
round_one_ids = [train_and_log(f"dt_r1_{index}", "Decision Tree", 1, parameters)
                 for index, parameters in enumerate(tree_candidates, start=1)]


## Task 6: Compare the first three runs

Run the comparison cell, then open **Experiments → seer_milestone_delay**. Use **List** to inspect parameters and metrics, and **Compare** to visualize `validation_f1` and `validation_brier_score`. The notebook table includes only the runs created in this walkthrough.

Do not choose a winner yet—Task 7 adds the Gradient Boosting runs.


In [ ]:
initial_comparison = compare_runs()
display(format_comparison(initial_comparison))


## Task 7: Round two — Gradient Boosting

Run the three Gradient Boosting candidates.

| Candidate | max_depth | learning_rate | n_estimators |
| --- | --- | --- |
| gb_r2_1 | 2 | 0.05 | 60 |
| gb_r2_2 | 3 | 0.10 | 100 |
| gb_r2_3 | 4 | 0.05 | 120 |

Keep the same experiment and session. These runs can now be compared with the Decision Tree runs.


In [ ]:
gb_candidates = [
    {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 60},
    {"max_depth": 3, "learning_rate": 0.10, "n_estimators": 100},
    {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 120},
]
round_two_ids = [train_and_log(f"gb_r2_{index}", "Gradient Boosting", 2, parameters)
                 for index, parameters in enumerate(gb_candidates, start=1)]


## Task 8: Compare all six and select the winner

Run the cell to compare all six candidates. The highest validation F1 wins; when F1 is tied, the lower validation Brier score wins.

You can make the same selection in **Experiments → seer_milestone_delay → Compare**: prefer higher `validation_f1`, then lower `validation_brier_score`.


In [ ]:
final_comparison = compare_runs()
display(format_comparison(final_comparison))

best_run = final_comparison.iloc[0]
best_run_id = best_run["run_id"]
best_model_uri = mlflow.get_run(best_run_id).data.tags["model_uri"]
print("Selected run:", best_run_id)
print("Selection rule: highest validation F1, then lowest validation Brier score")
print(
    f"Selected metrics: F1 = {best_run['metrics.validation_f1']:.4f}, "
    f"Brier score = {best_run['metrics.validation_brier_score']:.4f}"
)


## Task 9: Register the selected model

Choose exactly one route:

- **Notebook:** Run the cell once. It registers the selected F1/Brier winner as `seer_livelabs.seer_gold.milestone_delay_classifier`.
- **Experiments UI:** Open the selected run, choose **Register**, select its `model` artifact, and enter that same fully qualified name.

Do not use both routes for the same selection: each registration creates a model version. In **Master catalog → seer_livelabs → seer_gold → Models**, open `milestone_delay_classifier` and use the version's **Run** link to trace it to the selected experiment run.


In [ ]:
registered_version = mlflow.register_model(model_uri=best_model_uri, name=MODEL_NAME)
MODEL_VERSION = str(registered_version.version)
REGISTERED_URI = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
print("Registered model:", MODEL_NAME)
print("Version:", MODEL_VERSION, "| Source run:", best_run_id)
print("Versioned model URI:", REGISTERED_URI)


## Task 10: Load the registered version and score Gold milestones

Run the cell to load the exact catalog version created in Task 9 and score `seer_gold.milestone_features`.


In [ ]:
from pyspark.sql import functions as F

loaded_model = mlflow.sklearn.load_model(REGISTERED_URI)
scoring_rows = spark.table(SCORING_TABLE)
positive_class_index = list(loaded_model.classes_).index(1)
predict_proba = mlflow.pyfunc.spark_udf(
    spark, model_uri=REGISTERED_URI, result_type="array<double>", env_manager="local")

predictions = scoring_rows.select(
    "milestone_id", "project_name", "milestone_name", "planned_date",
    predict_proba(F.struct(*[F.col(name).cast("double").alias(name) for name in FEATURES]))[
        positive_class_index
    ].alias("delay_probability"),
).withColumn("predicted_late", F.col("delay_probability") >= threshold) \
 .withColumn("model_version", F.lit(MODEL_VERSION))

print("Scoring with:", REGISTERED_URI)
display(predictions.orderBy(F.desc("delay_probability")))
